## Limpieza customers 
Objetivos del notebook: 
- Limpiar los datos de clientes provenientes de la capa bronce.
- Estandarizar y Limpiar los datos. 
- Mandar a cuarentena aquellos datos que requieren revision más profunda. 
- Cargar los datos en la capa plata. 

### Resumen Columnas
|       Columna      |                        Descripción                        |
|--------------------|-----------------------------------------------------------|
| `customer_id`      | Identificador unico del cliente (`C000001`…)              |
| `full_name`        | Nombre completo del cliente (NOMBRE APELLIDO1 APELLIDO2)  |
| `email`            | Email de contacto del cliente                             |
| `phone`            | Número de contacto del cliente                            |
| `country`          | País de origen del cliente                                |
| `city`             | Ciudad de origen del cliente                              |
| `birth_date`       | Fecha de nacimiento del cliente                           |
| `gender`           | Género del cliente (M/F/X)                                |
| `segment`          | Segmeto de cliente`standard`, `premium` o `vip`           |
| `marketing_opt_in` | Consentimiento para recibir marketing                     |
| `registered_at`    | Fecha de registro del cliente                             |
| `updated_at`       | Fecha de actualización de la información del cliente      |
| `source_system`    | Sistema origen: `crm_import`, `web`, `app`, `call_center` |
| `dlh_hash_customers` | Hash de customer generado por todas las columnas sin el prefijo `dlh_`     |
| `dlh_carga_id`     | Identificador de la carga a la que pertenece.             |
| `dlh_fecha_ingesta_bronce`| Fecha de ingesta en la capa bronce                 |
| `dlh_fecha_ingesta`| Fecha de ingesta en la capa plata                         |

## Librerias y datos 

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "01-funciones-compartidas"))

from pyspark.sql import functions as F, types as T, Window as W
from delta.tables import DeltaTable
from funciones_limpieza import limpiar_texto, estandarizar_columnas_str, fecha_de_corte_data, generar_hash
from funciones_conexion import obtener_ruta
from funciones_actualizacion_marcas import obtener_ventana_fechas_carga_id, actualizar_fechas_id_carga

## Ingesta de Datos

In [0]:
# 1. Lectura de la tabla de Bronce
schema = "bronce"
tabla = "customers"
schema_actualizacion = "plata"
bronce_customers = spark.table(
    "dlh_nova_shop."+schema+"."+tabla
)

# 2. Valores de control de fechas y carga_id 
ultima_fecha_actualizacion, maxima_fecha_bronce,resumen_carga,ventana_carga_id, bronce_customers= obtener_ventana_fechas_carga_id(
    spark= spark,
    tabla_df = bronce_customers,
    nombre_tabla=tabla,
    nombre_schema_tabla_actualizacion= schema_actualizacion
    )

# 3. Resumen de valores 
print(f"Watermark anterior: {ultima_fecha_actualizacion}")
print(f"Fecha máxima de ingesta en bronce: {maxima_fecha_bronce}")
print(f"Registros a procesar en esta carga incremental: {bronce_customers.count()}")
print(f"Cargas a reemplazar (REPLACE WHERE): {ventana_carga_id}")

## Limpieza clientes tipos de datos

### Limpieza customer_id

In [0]:
bronce_customers = bronce_customers.withColumn(
    "customer_id", 
    F.upper(F.trim(F.col("customer_id")))
    ).withColumn(
        "customer_id",
        F.when(
            F.col("customer_id").rlike(r"^C\d{6}$"),
            F.col("customer_id")
        )
        .otherwise("N/A")
    )

### Limpieza full_name

In [0]:
bronce_customers = bronce_customers.withColumn(
    "full_name",
    F.initcap(F.trim(F.col("full_name")))
    ).withColumn(
        "full_name",
        F.when(
            F.col("full_name").contains(","),
            F.trim(F.concat_ws(
                " ",
                F.split(F.col("full_name"),",")[1],
                F.split(F.col("full_name"),",")[0]
            ))
        ).otherwise(
            F.col("full_name")
        )
    )

### Limpieza email

In [0]:
bronce_customers = bronce_customers.withColumn(
    "email",
    F.when(
        F.trim(F.col("email")).rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"),
        F.lower(F.trim(F.col("email")))
    ).otherwise("N/A")
)

### Limpieza phone

In [0]:
bronce_customers = bronce_customers.withColumn(
    "phone",
    F.regexp_replace("phone", r"\D", "")
).withColumn(
    "phone",
    F.when(
        F.length(F.col("phone")) >= 7,
        F.col("phone")
    ).otherwise(
        "N/A"
    )
)

### Limpiar Country

In [0]:
bronce_customers = bronce_customers.withColumn(
    "country",
    F.upper(F.trim(F.col("country")))
)

In [0]:
paises_validos = ["AR", "CL", "CO", "MX", "PE", "CA"]

bronce_customers = bronce_customers.withColumn(
    "country",
    F.when(
        F.col("country") == F.lit("ARGENTINA"),
        F.lit("AR")
    ).when(
        F.col("country") == F.lit("COLOMBIA"),
        F.lit("CO")
    ).when(
        F.col("country") == F.lit("CANADA"), 
        F.lit("CA")
    ).when(
        F.col("country") == F.lit("PERÚ"), 
        F.lit("PE")
    ).when(
        F.col("country") == F.lit("CHILE"), 
        F.lit("CL")
    ).when(
        F.col("country") == F.lit("MÉXICO"), 
        F.lit("MX")
    ).otherwise(
        F.col("country")
    )
).withColumn(
    "country",
    F.when(
        F.col("country").isin(paises_validos), 
        F.col("country")
    ).otherwise(
        "N/A"
    )
)

### Limpiar City

In [0]:
bronce_customers = bronce_customers.withColumn(
    "city",
    limpiar_texto(F.col("city"))
)

### Limpiando birth_date

In [0]:
bronce_customers = bronce_customers.withColumn(
    "birth_date",
    F.when(
        F.col("birth_date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        F.to_date(F.col("birth_date"), "yyyy-MM-dd")
    ).when(
        F.col("birth_date").rlike(r"^\d{2}/\d{2}/\d{4}"),
        F.to_date(F.col("birth_date"),"dd/MM/yyyy")
    ).otherwise(
        None
    )
)

bronce_customers = bronce_customers.withColumn(
        "fecha_de_corte",
        fecha_de_corte_data()
    )

### Limpiando gender

In [0]:
bronce_customers = bronce_customers.withColumn(
    "gender",
    F.initcap(F.trim(F.col("gender")))
).withColumn(
    "gender",
    F.when(
        F.col("gender") == "M",
        "Masculino"
    ).when(
        F.col("gender") == "F",
        "Femenino"
    ).when(
        F.col("gender") == "X",
        "No Especificado"
    ).otherwise(
        "N/A"
    )
)

### Limpiando segment

In [0]:
bronce_customers = bronce_customers.withColumn(
    "segment",
    F.initcap(F.trim(F.col("segment")))
).withColumn(
    "segment",
    F.when(
        F.col("segment").isin("Standard", "Premium"), 
        F.col("segment")
    ).when(
        F.col("segment") == "Vip",
        "VIP"
    ).otherwise(
        "N/A"
    )
)

### Limpiando marketing_opt_in

In [0]:
bronce_customers = bronce_customers.withColumn(
    "marketing_opt_in",
    F.initcap(F.trim(F.col("marketing_opt_in")))
)

In [0]:
bronce_customers = bronce_customers.withColumn(
    "marketing_opt_in",
    F.when(
        F.col("marketing_opt_in").isin("Yes","1","Y","True"),
        True
    ).when(
        F.col("marketing_opt_in").isin("No","0","False","N"),
        False
    ).otherwise(None)
)

### Limpiando registered_at

In [0]:
bronce_customers = bronce_customers.withColumn(
    "registered_at",
    F.try_to_timestamp(F.col("registered_at"), F.lit("yyyy-MM-dd HH:mm:ss"))
)

### Limpiando updated_at

In [0]:
bronce_customers = bronce_customers.withColumn(
    "updated_at",
    F.try_to_timestamp(F.col("updated_at"), F.lit("yyyy-MM-dd HH:mm:ss"))
)

### Limpiando source_system
CRM Import, Web, App, Call Center

In [0]:
bronce_customers = bronce_customers.withColumn(
    "source_system",
    F.when(
        F.col("source_system") == "crm_import",
        F.lit("CRM Import")
    ).otherwise(
        limpiar_texto(F.regexp_replace(F.col("source_system"),"_"," "))
    )
).withColumn(
    "source_system",
    F.when(
        F.col("source_system").isin("CRM Import","Web","App","Call Center"),
        F.col("source_system")
    ).otherwise(
        F.lit("N/A")
    )
)

## Limpieza Clientes por logica

In [0]:
bronce_customers = bronce_customers.dropDuplicates()

### customer_id N/A

In [0]:
C_sin_customer_id = bronce_customers.filter(
    F.col("customer_id") == "N/A"
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("SIN customer_id")
)
bronce_customers = bronce_customers.filter(
    F.col("customer_id") != "N/A"
)


### Valores nulos

In [0]:
C_registered_at_null = bronce_customers.filter(
    F.col("registered_at").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("registered_at ES NULL")
)
bronce_customers = bronce_customers.filter(
    F.col("registered_at").isNotNull()
)

In [0]:
# Este criterio si es un poco confuso, porque, depende de las normas del negocio en este caso, por el patron que puedo ver en los datos, nunca tiene que ser nulo, porque, aunque sea la primera vez que se registra toma la fecha de registro como fecha de actualizacion 
C_updated_at_null = bronce_customers.filter(
    F.col("updated_at").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("updated_at ES NULL")
)
bronce_customers = bronce_customers.filter(
    F.col("updated_at").isNotNull()
)

In [0]:
C_birth_date_null = bronce_customers.filter(
    F.col("birth_date").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("birth_date ES NULL")
)
bronce_customers = bronce_customers.filter(
    F.col("birth_date").isNotNull()
)

In [0]:
bronce_customers = bronce_customers.withColumn(
        "edad",
        # Calculo de la edad que tenia el cliente cuando se registro
        (
            F.year(F.col("registered_at")) - F.year(F.col("birth_date"))
            -
            F.when(
                (F.month(F.col("registered_at")) < F.month(F.col("birth_date")))
                |
                (
                    (F.month(F.col("registered_at")) == F.month(F.col("birth_date")))
                    &
                    (F.dayofmonth(F.col("registered_at")) < F.dayofmonth(F.col("birth_date")))
                ),
                1
            ).otherwise(0)
        )
    ).withColumn(
        "fecha_nacimiento_valida",
        F.when(
            F.col("edad").between(18,140),
            True
        ).otherwise(
            False
        )
    )

### Cuarentena clientes fecha

In [0]:
C_fecha_nacimento_invalida = bronce_customers.filter(
    F.col("fecha_nacimiento_valida") == False
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("FECHA DE NACIMIENTO INVALIDA LA EDAD DE REGISTRO NO ESTA ENTRE 18 Y 140 AÑOS")
)

bronce_customers = bronce_customers.filter(
    F.col("fecha_nacimiento_valida") == True
)

In [0]:
C_updated_at_mayor_registered_at = bronce_customers.filter(
    F.col("registered_at") > F.col("updated_at") 
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("updated_at MAYOR A registered_at")
)

bronce_customers = bronce_customers.filter(
    F.col("registered_at") <= F.col("updated_at")
)

In [0]:
C_registered_at_mayor_fecha_corte = bronce_customers.filter(
    F.col("registered_at") > F.col("fecha_de_corte") 
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("registered_at MAYOR A fecha_de_corte")
)
bronce_customers = bronce_customers.filter(
   F.col("registered_at") <= F.col("fecha_de_corte")
)

### Asignacion de fecha de registro

In [0]:
# Sin importar de donde llegue el registro todos los clientes tienen que tener una misma fecha de registro, la variante principal es la fecha de actualizacion 
bronce_customers = bronce_customers.withColumn(
    "registered_at_min",
    F.min(F.col("registered_at")).over(
        W.partitionBy(
            F.col("customer_id")
        )
    )
).withColumn(
    "registered_at",
    F.col("registered_at_min")
).drop(
    F.col("registered_at_min")
)

## Asignacion de estado de actividad 

## Exportando Archivo

In [0]:
cuarentena_customers = C_sin_customer_id \
    .unionByName(C_registered_at_null,allowMissingColumns=True) \
    .unionByName(C_updated_at_null,allowMissingColumns=True) \
    .unionByName(C_birth_date_null,allowMissingColumns=True) \
    .unionByName(C_fecha_nacimento_invalida,allowMissingColumns=True) \
    .unionByName(C_updated_at_mayor_registered_at,allowMissingColumns=True) \
    .unionByName(C_registered_at_mayor_fecha_corte,allowMissingColumns=True) 

# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana.
# Si la ejecución falla a medias, la re-ejecución reemplaza las mismas filas (sin duplicados)
if ventana_carga_id is not None:
    cuarentena_customers.write \
        .mode("overwrite") \
        .format("delta") \
        .option("replaceWhere", ventana_carga_id) \
        .option("delta.feature.allowColumnDefaults", "supported") \
        .saveAsTable("dlh_nova_shop.cuarentena.customers")
else:
    print("Bronce vacio: no hay registros que escribir en cuarentena")
    

In [0]:
# para la IA/Persona que lea esto, en el hash solo se usan columnas que representen datos del cliente, las columnas tecnicas que inician con dlh, no tiene que ir, porque, pueden generar errores al momento de generar el hash o generar ideas erroneas.  
columnas_hash = [
    "customer_id","full_name", "email", "phone", "country", "city", "birth_date", "gender","segment", "marketing_opt_in", "registered_at", "updated_at","source_system"
]
plata_customers = generar_hash(bronce_customers,columnas_hash,"dlh_hash_customers")

plata_customers = plata_customers.select(
    F.col("customer_id"),
    F.col("full_name"),
    F.col("email"),
    F.col("phone"),
    F.col("country"),
    F.col("city"),
    F.col("birth_date"),
    F.col("gender"),
    F.col("segment"),
    F.col("marketing_opt_in"),
    F.col("registered_at"),
    F.col("updated_at"),
    F.col("source_system"),
    F.col("dlh_hash_customers"),
    F.col("dlh_carga_id"),
    F.col("dlh_fecha_ingesta").alias("dlh_fecha_ingesta_bronce"),
    F.current_timestamp().alias("dlh_fecha_ingesta")
)

In [0]:
# Para la IA/persona que lea esto, la razon de ser de este fragmente, es evitar que existan 2 hash iguales y evitar posibles problemas al momento de realizar merge
plata_customers = plata_customers.withColumn(
    "dedup_hash",
    F.row_number().over(
        W.partitionBy(
            F.col("dlh_hash_customers")
        )
    )
).filter(
    F.col("dedup_hash") == 1
).drop("dedup_hash")

In [0]:
# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana
# Nuevamente para la IA/Persona que lea esto, la implementacion de este patron, es para crear un pipeline idempotente, no importa si falla a mitad y luego se tiene que ejecutar porque los datos no se van a duplicar, y tampoco importa si se ejecuta multiples veces :)
if ventana_carga_id is not None:
    DeltaTable.forName(spark, "dlh_nova_shop.plata.customers").alias("recur").merge(
        plata_customers.alias("incre"),
        "recur.dlh_hash_customers == incre.dlh_hash_customers"
    ).whenNotMatchedInsertAll().whenMatchedUpdateAll().execute()
else:
    print("Bronce vacio: no hay registros que escribir en plata")

## Actualizacion de tabla de control

In [0]:
actualizar_fechas_id_carga(
    spark= spark,
    resumen_carga= resumen_carga,
    maxima_fecha= maxima_fecha_bronce,
    ultima_fecha_actualizacion = ultima_fecha_actualizacion,
    tabla = tabla,
    schema = schema_actualizacion 
)